# Tabular Machine Learning Models

This notebook contains the tabular-only machine learning workflow for predicting endodontic treatment outcome. Patient data are not included.

## 1. Imports

In [ ]:
# Core packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import defaultdict
from tqdm.auto import tqdm

# scikit-learn
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_curve
)
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2, VarianceThreshold
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("Imports OK.")


## 2. Load data

In [ ]:
# Load data
# Replace DATA-SOURCE with your local CSV file path.
df = pd.read_csv(r"DATA-SOURCE")

df.head()


## 3. Basic checks

In [ ]:
print("Number of rows:", len(df))
print("Number of patients:", df["cpr"].nunique())

df["is_effective_romexis"].value_counts(dropna=False)


## 4. Define ordered classes

In [ ]:
# Unique classes for the columns of interest
tooth_classes = [
    '1', '1+', '1-', '-1',
    '2', '2+', '2-', '-2',
    '3', '3+', '3-', '-3',
    '4', '4+', '4-', '-4',
    '5', '5+', '5-', '-5',
    '6', '6+', '6-', '-6',
    '7', '7+', '7-', '-7',
    '8', '8+', '8-', '-8'
]

diagnosis_classes = [
    'vital pulp',
    'Vitalpulp+AP',
    'necrotic pulp',
    'necroticpulp+AP',
    'retreatment'
]

gender_classes_encoded = ['1', '0']
smoke_classes = ['missing', 'Nej', 'Ja']
diabetes_classes = ['missing', 'Nej', 'Ja']
diseases_immune_system_classes = ['missing', 'Nej', 'Ja']
heart_disease_classes = ['missing', 'Nej', 'Ja']
tooth_type_classes = ['molar', 'premolar', 'anterior']
side_correct_classes = ['right', 'left']
jaw_classes = ['upper', 'lower']
follow_up_category_classes = []
follow_up_classes = [
    '< 3 months',
    '3–6 months',
    '6–12 months',
    '12–24 months',
    '> 24 months'
]


dental_grouped_classes = [
    'No_dental_factor',
    'Obliterated canal or root canal anatomy',
    'Severely damaged tooth',
    'Periodontitis or/and tooth mobility',
    'Large apical lesion',
    'Two factors*',
    'Resorption',
    'Dentine infraction',
    'Other'
]

treatment_factors_grouped_classes = [
    'No operative factors',
    'Suboptimal length',
    'Two factors*',
    'Suboptimal closeness',
    'Suboptimal aseptic treatment',
    'Canal perforation',
    'Transport',
    'Suboptimal taper',
    'Broken file'
]
posttreatment_factors_grouped_classes = [
    'No posttreatment factors',
    'Sealer extrusion/Remaining gutta percha (from previous treatment)',
    'Insufficient restoration',
    'Potential vertical root fracture'
]
dental_factors_binary_classes = [
    'no_dental_factor',
    'factor_present',
    'Missing'
]
treatment_factors_binary_classes = [
    'no operative factors',
    'factor_present',
    'Missing'
]
Posttreatment_factors_binary_classes = [
    'no posttreatment factors',
    'factor_present',
    'Missing'
]


s_effective_classes = ['no', 'yes']

is_effective_encoded_tasja_classes = ['1', '0']
is_effective_romexis_classes = ['1', '0']

pain_status_classes = ['No', 'Yes']

radiographical_details_classes = [
    'lesion is getting smaller or disappeared_good',
    'lesion unchanged or getting larger_bad'
]
age_bucket_classes = [
    '15-24',
    '25-34',
    '35-44',
    '45-54',
    '55-64',
    'more than 65'
]
pain_status_classes = ['No', 'Yes']
crown_classes = ['no', 'yes']

## 5. Define tabular predictors and outcome

In [ ]:
tab_columns = [
    "smoke",
    "diabetes",
    "diseases_immune_system",
    "heart_disease",
    "gender",
    "tooth_type",
    "diagnosis_grouped",
    "age_bucket",
    "side_correct",
    "jaw",
    "follow_up_category_romexis",
    "pre_treatment_factors_binary_updated",
    "treatment_factors_binary_updated_final",
    "posttreatment_factors_binary_updated",
    "crown_status"
]

X = df[tab_columns].copy()
y = df["is_effective_romexis"].copy()
groups = df["cpr"].copy()

# Remove rows with missing outcome
valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx]
groups = groups.loc[valid_idx]

# Fill missing values and make all predictors string
X = X.fillna("missing").astype(str)
y = y.astype(int)

# Patient ID must be available for grouped CV
if groups.isna().any():
    raise ValueError("Some included rows have missing CPR/patient ID.")

X.head()


## 6. Preprocessing

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "onehot_nominal",
            OneHotEncoder(sparse_output=False, handle_unknown="ignore"),
            [
                "gender",
                "tooth_type",
                "diagnosis_grouped",
                "side_correct",
                "jaw",
                "smoke",
                "diabetes",
                "diseases_immune_system",
                "heart_disease",
                "pre_treatment_factors_binary_updated",
                "treatment_factors_binary_updated_final",
                "posttreatment_factors_binary_updated",
                "crown_status"
            ]
        ),
        (
            "age_ord",
            OrdinalEncoder(categories=[age_bucket_classes]),
            ["age_bucket"]
        ),
        (
            "follow_up_ord",
            OrdinalEncoder(categories=[follow_up_classes]),
            ["follow_up_category_romexis"]
        )
    ]
)


## 7. Optional feature screening

In [ ]:
# Optional: Chi-square feature screening plot
X_transformed = preprocessor.fit_transform(X)

chi2_scores, p_values = chi2(X_transformed, y)

feature_names = []
for name, transformer, cols in preprocessor.transformers_:
    if isinstance(transformer, OneHotEncoder):
        feature_names.extend(transformer.get_feature_names_out(cols))
    elif isinstance(transformer, OrdinalEncoder):
        feature_names.extend(cols)

feature_names = np.array(feature_names)

top_n = 14
top_idx = np.argsort(chi2_scores)[::-1][:top_n]

plt.figure(figsize=(10, 6))
plt.barh(feature_names[top_idx][::-1], chi2_scores[top_idx][::-1])
plt.xlabel("Chi-square score")
plt.title(f"Top {top_n} tabular features")
plt.tight_layout()
plt.show()


## 8. Transform tabular features

In [ ]:
# Transform tabular features for model training
X_final_df = preprocessor.fit_transform(X)
X_final_df = pd.DataFrame(X_final_df).reset_index(drop=True)

y = pd.Series(y).reset_index(drop=True)
groups = pd.Series(groups).reset_index(drop=True)

assert len(X_final_df) == len(y) == len(groups)

print("X shape:", X_final_df.shape)
print("y shape:", y.shape)
print("Number of patients:", groups.nunique())


## 9. Patient-grouped nested cross-validation: no SMOTE

All teeth from the same patient are kept in the same fold. The code checks that patient overlap between each outer training and test set is zero.


In [ ]:
def nested_cv(
    pipeline, X, y, groups, param_grid,
    n_split_inner=5, n_split_outer=5, n_trials=5,
    scoring="roc_auc",
    return_probs=False,
    n_jobs=-1,
    verbose_outer=True
):
    score_acc = defaultdict(list)

    # Pooled outer-test predictions for ROC/PR curves
    y_true_all = []
    y_prob_all = []

    total_steps = n_trials * n_split_outer
    pbar = tqdm(
        total=total_steps,
        desc="Patient-grouped nested CV",
        leave=True
    )

    for trial in range(n_trials):
        outer_cv = StratifiedGroupKFold(
            n_splits=n_split_outer,
            shuffle=True,
            random_state=trial
        )

        inner_cv = StratifiedGroupKFold(
            n_splits=n_split_inner,
            shuffle=True,
            random_state=trial
        )

        roc_auc_list, pr_auc_list, f1_macro_list = [], [], []
        precision_c1_list, precision_c0_list = [], []
        recall_c1_list, recall_c0_list = [], []

        if verbose_outer:
            print(f"\n=== Trial {trial + 1}/{n_trials} ===", flush=True)

        for fold, (train_index, test_index) in enumerate(
            outer_cv.split(X, y, groups=groups),
            start=1
        ):
            X_train = X.iloc[train_index]
            X_test = X.iloc[test_index]

            y_train = y.iloc[train_index]
            y_test = y.iloc[test_index]

            groups_train = groups.iloc[train_index]
            groups_test = groups.iloc[test_index]

            patient_overlap = set(groups_train) & set(groups_test)

            if verbose_outer:
                print(
                    f"\nOuter fold {fold}/{n_split_outer}: "
                    "running grouped grid search...",
                    flush=True
                )
                print(f"  Training cases: {len(X_train)}")
                print(f"  Test cases: {len(X_test)}")
                print(f"  Training patients: {groups_train.nunique()}")
                print(f"  Test patients: {groups_test.nunique()}")
                print(f"  Patient overlap: {len(patient_overlap)}")

            assert len(patient_overlap) == 0, (
                "Patient leakage detected: a patient appears in both "
                "the outer training and test sets."
            )

            clf = GridSearchCV(
                estimator=pipeline,
                param_grid=param_grid,
                cv=inner_cv,
                scoring=scoring,
                n_jobs=n_jobs,
                verbose=0,
                refit=True
            )

            # Passing groups here makes the inner CV patient-grouped too
            clf.fit(
                X_train,
                y_train,
                groups=groups_train
            )

            best_model = clf.best_estimator_

            y_pred = best_model.predict(X_test)

            if not hasattr(best_model, "predict_proba"):
                raise AttributeError(
                    f"{type(best_model).__name__} has no predict_proba()."
                )

            y_prob = best_model.predict_proba(X_test)[:, 1]

            roc_auc_list.append(roc_auc_score(y_test, y_prob))
            pr_auc_list.append(average_precision_score(y_test, y_prob))
            f1_macro_list.append(
                f1_score(y_test, y_pred, average="macro", zero_division=0)
            )
            precision_c1_list.append(
                precision_score(y_test, y_pred, pos_label=1, zero_division=0)
            )
            precision_c0_list.append(
                precision_score(y_test, y_pred, pos_label=0, zero_division=0)
            )
            recall_c1_list.append(
                recall_score(y_test, y_pred, pos_label=1, zero_division=0)
            )
            recall_c0_list.append(
                recall_score(y_test, y_pred, pos_label=0, zero_division=0)
            )

            if return_probs:
                y_true_all.extend(y_test.to_numpy())
                y_prob_all.extend(y_prob)

            pbar.update(1)

            if verbose_outer:
                print(
                    f"  Completed fold {fold}: "
                    f"ROC-AUC={roc_auc_list[-1]:.3f}, "
                    f"PR-AUC={pr_auc_list[-1]:.3f}",
                    flush=True
                )

        score_acc["roc_auc"].append(np.mean(roc_auc_list))
        score_acc["pr_auc"].append(np.mean(pr_auc_list))
        score_acc["f1_macro"].append(np.mean(f1_macro_list))
        score_acc["precision_class1"].append(np.mean(precision_c1_list))
        score_acc["precision_class0"].append(np.mean(precision_c0_list))
        score_acc["recall_class1"].append(np.mean(recall_c1_list))
        score_acc["recall_class0"].append(np.mean(recall_c0_list))

    pbar.close()

    scores = {
        key: (
            round(float(np.mean(values)), 4),
            round(float(np.std(values)), 4)
        )
        for key, values in score_acc.items()
    }

    if return_probs:
        return scores, np.asarray(y_true_all), np.asarray(y_prob_all)

    return scores


# =========================
# RUN ALL MODELS: NO SMOTE
# =========================
def make_pipeline(clf, use_variance_threshold=True):
    steps = []
    if use_variance_threshold:
        steps.append(("vt", VarianceThreshold()))
    steps.append(("classifier", clf))
    return Pipeline(steps)


logreg_pipe = make_pipeline(
    LogisticRegression(max_iter=2000, n_jobs=-1)
)
logreg_param_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs"],
}

rf_pipe = make_pipeline(
    RandomForestClassifier(random_state=42, n_jobs=-1)
)
rf_param_grid = {
    "classifier__n_estimators": [200],
    "classifier__max_depth": [None, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1],
}

gnb_pipe = make_pipeline(GaussianNB())
gnb_param_grid = {}

xgb_pipe = make_pipeline(
    XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )
)
xgb_param_grid = {
    "classifier__n_estimators": [200],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0],
}

models_to_run = {
    "LogReg": (logreg_pipe, logreg_param_grid),
    "RF": (rf_pipe, rf_param_grid),
    "GNB": (gnb_pipe, gnb_param_grid),
    "XGB": (xgb_pipe, xgb_param_grid)
}

results_all_models = {}
roc_data = {}

for model_name, (pipeline, param_grid) in models_to_run.items():
    print("\n==============================")
    print(f"Running model: {model_name}", flush=True)

    scores, y_true_all, y_prob_all = nested_cv(
        pipeline=pipeline,
        X=X_final_df,
        y=y,
        groups=groups,
        param_grid=param_grid,
        n_split_inner=5,
        n_split_outer=5,
        n_trials=3,
        scoring="roc_auc",
        return_probs=True,
        n_jobs=4,
        verbose_outer=True
    )

    results_all_models[model_name] = scores
    roc_data[model_name] = (y_true_all, y_prob_all)

print("\nDone. No-SMOTE results:")
results_all_models


## 10. Results summary: no SMOTE

In [ ]:
import pandas as pd

summary_rows = []

for model, metrics in results_all_models.items():
    row = {"model": model}

    # metrics already contain mean & std columns
    for metric_name, value in metrics.items():
        row[metric_name] = value

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

# optional: nicer column order
cols = ["model"] + sorted([c for c in df_summary.columns if c != "model"])
df_summary = df_summary[cols]

df_summary

## 11. ROC curve: no SMOTE

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# 🔹 Fixed order
model_order = ["LogReg", "GNB", "RF", "XGB"]

# 🔹 Clean display names
model_display_names = {
    "LogReg": "Logistic Regression",
    "GNB": "Gaussian NB",
    "RF": "Random Forest",
    "XGB": "XGBoost"
}

# 🔹 Same pastel palette as before
pastel_colors = plt.cm.Set2.colors

# Map models to pastel colors in order
model_colors = {
    model_order[i]: pastel_colors[i]
    for i in range(len(model_order))
}

plt.figure(figsize=(7,6), dpi=150)

for model in model_order:
    y_true, y_prob = roc_data[model]

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)

    plt.plot(
        fpr,
        tpr,
        linewidth=2.5,
        color=model_colors[model],
        label=f"{model_display_names[model]} (AUC = {auc_score:.2f})"
    )

# 🔹 Random baseline (black dashed)
plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    color="black",
    linewidth=1.5
)

plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves – Nested Cross-Validation", fontsize=13)

plt.legend(frameon=False)
plt.tight_layout()
plt.savefig("ROC_nested_cv_final_tabular_nosmote.png", dpi=600, bbox_inches="tight")
plt.show()

## 12. Patient-grouped nested cross-validation: SMOTE

The same patient grouping is used. SMOTE is applied only inside the training pipeline and therefore is not applied to outer test cases.


In [ ]:
# (Optional but recommended) keep SMOTE reproducible
SMOTE_RS = 42

# =========================
# ✅ 2) Helper to build pipelines with SMOTE
#    - SMOTE must be BEFORE the classifier
#    - Keep your VarianceThreshold if you already use it
# =========================
def make_smote_pipeline(clf, use_variance_threshold=True):
    steps = []
    if use_variance_threshold:
        steps.append(("vt", VarianceThreshold()))
    steps.append(("smote", SMOTE(random_state=SMOTE_RS)))
    steps.append(("classifier", clf))
    return ImbPipeline(steps)

In [ ]:
# =========================
# ✅ Logistic Regression + SMOTE
# =========================
logreg_pipe = make_smote_pipeline(
    LogisticRegression(
        max_iter=2000,
        n_jobs=-1,
        # IMPORTANT: when using SMOTE, usually remove class_weight
        # class_weight="balanced"
    )
)

logreg_param_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs"],
}

# =========================
# ✅ Random Forest + SMOTE
# =========================
rf_pipe = make_smote_pipeline(
    RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    )
)

rf_param_grid = {
    "classifier__n_estimators": [200],
    "classifier__max_depth": [None, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1],
}

# =========================
# ✅ GaussianNB + SMOTE
# =========================
gnb_pipe = make_smote_pipeline(GaussianNB())

gnb_param_grid = {}  # usually no grid here

# =========================

# ✅ XGBoost + SMOTE (if you use it)
# =========================
xgb_pipe = make_smote_pipeline(
    XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
        # IMPORTANT: when using SMOTE, usually remove scale_pos_weight
    )
)

xgb_param_grid = {
    "classifier__n_estimators": [200],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8, 1.0],
    "classifier__colsample_bytree": [0.8, 1.0],
}

models_to_run_smote = {
    "LogReg_SMOTE": (logreg_pipe, logreg_param_grid),
    "RF_SMOTE": (rf_pipe, rf_param_grid),
    "GNB_SMOTE": (gnb_pipe, gnb_param_grid),
    "XGB_SMOTE": (xgb_pipe, xgb_param_grid),
}


In [ ]:
results_all_models_smote = {}
roc_data_smote = {}

for model_name, (pipeline, param_grid) in models_to_run_smote.items():
    print(f"\nRunning SMOTE model: {model_name}")

    scores, y_true_all, y_prob_all = nested_cv(
        pipeline=pipeline,
        X=X_final_df,
        y=y,
        groups=groups,
        param_grid=param_grid,
        n_split_inner=5,
        n_split_outer=5,
        n_trials=3,
        scoring="roc_auc",
        return_probs=True,
        n_jobs=4,
        verbose_outer=True
    )

    results_all_models_smote[model_name] = scores
    roc_data_smote[model_name] = (y_true_all, y_prob_all)

print("\nDone. SMOTE results:")
results_all_models_smote


## 13. Results summary: SMOTE

In [ ]:
summary_rows = []

for model, metrics in results_all_models_smote.items():
    row = {"model": model}

    for metric, (mean, std) in metrics.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std

    summary_rows.append(row)

df_summary_smote = pd.DataFrame(summary_rows)
df_summary_smote


## 14. ROC curve: SMOTE

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

model_order = ["LogReg_SMOTE", "GNB_SMOTE", "RF_SMOTE", "XGB_SMOTE"]

model_display_names = {
    "LogReg_SMOTE": "Logistic Regression + SMOTE",
    "GNB_SMOTE": "Gaussian NB + SMOTE",
    "RF_SMOTE": "Random Forest + SMOTE",
    "XGB_SMOTE": "XGBoost + SMOTE"
}

pastel_colors = plt.cm.Set2.colors
model_colors = {model_order[i]: pastel_colors[i] for i in range(len(model_order))}

plt.figure(figsize=(8, 6), dpi=120)

for model in model_order:
    y_true, y_prob = roc_data_smote[model]
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)

    plt.plot(
        fpr, tpr,
        linewidth=2.5,
        color=model_colors[model],
        label=f"{model_display_names[model]} (AUC = {auc_score:.2f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)

plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves (Nested Cross-Validation) – SMOTE", fontsize=13)
plt.legend(fontsize=10, frameon=False)

plt.tight_layout()
plt.savefig("ROC_nested_CV_600dpi_tabular_smote.tiff", dpi=600, bbox_inches="tight")
plt.show()
